In [1]:
import cv2
import numpy as np
import torch

from pathlib import Path


ImageNet_Mean = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32
)

ImageNet_Std = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32
)


def Load_And_Preprocess_Video(
    Video_Path,
    Number_Of_Frames=16,
    Image_Size=224
):

    Video_Path = Path(
        Video_Path
    )

    if not Video_Path.exists():
        raise FileNotFoundError(
            f"Video not found: {Video_Path}"
        )

    Video_Capture = cv2.VideoCapture(
        str(Video_Path)
    )

    if not Video_Capture.isOpened():
        raise RuntimeError(
            f"Cannot open video: {Video_Path}"
        )

    Frame_Count = int(
        Video_Capture.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    if Frame_Count <= 0:
        Video_Capture.release()

        raise RuntimeError(
            f"Invalid frame count: {Video_Path}"
        )

    Frame_Indices = np.linspace(
        0,
        Frame_Count - 1,
        Number_Of_Frames,
        dtype=int
    )

    Processed_Frames = []

    for Frame_Index in Frame_Indices:

        Video_Capture.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(Frame_Index)
        )

        Success, Frame = (
            Video_Capture.read()
        )

        if not Success:
            Video_Capture.release()

            raise RuntimeError(
                f"Cannot read frame "
                f"{Frame_Index} from "
                f"{Video_Path}"
            )

        Frame_RGB = cv2.cvtColor(
            Frame,
            cv2.COLOR_BGR2RGB
        )

        Gray_Frame = cv2.cvtColor(
            Frame_RGB,
            cv2.COLOR_RGB2GRAY
        )

        Resized_Gray_Frame = cv2.resize(
            Gray_Frame,
            (
                Image_Size,
                Image_Size
            ),
            interpolation=cv2.INTER_LINEAR
        )

        Frame_3Channel = np.repeat(
            Resized_Gray_Frame[
                :,
                :,
                np.newaxis
            ],
            3,
            axis=2
        )

        Processed_Frames.append(
            Frame_3Channel
        )

    Video_Capture.release()

    Processed_Frames = np.stack(
        Processed_Frames,
        axis=0
    )

    Normalized_Frames = (
        Processed_Frames.astype(
            np.float32
        )
        /
        255.0
    )

    Normalized_Frames = (
        Normalized_Frames
        -
        ImageNet_Mean
    ) / ImageNet_Std

    Video_Tensor = torch.from_numpy(
        Normalized_Frames
    )

    Video_Tensor = (
        Video_Tensor
        .permute(
            3,
            0,
            1,
            2
        )
        .contiguous()
    )

    Expected_Shape = (
        3,
        Number_Of_Frames,
        Image_Size,
        Image_Size
    )

    if Video_Tensor.shape != Expected_Shape:
        raise RuntimeError(
            f"Unexpected video shape: "
            f"{Video_Tensor.shape}"
        )

    if not torch.isfinite(
        Video_Tensor
    ).all():
        raise RuntimeError(
            f"NaN or Inf found in: "
            f"{Video_Path}"
        )

    return (
        Video_Tensor,
        Frame_Indices
    )


print(
    "Preprocessing function loaded successfully."
)

Preprocessing function loaded successfully.


In [2]:
import pandas as pd

Echonet_Dir = Path(
    "/kaggle/input/datasets/mahnurrahman/"
    "echonet-dynamic/EchoNet-Dynamic"
)

Videos_Dir = (
    Echonet_Dir
    /
    "Videos"
)

FileList_Path = (
    Echonet_Dir
    /
    "FileList.csv"
)

FileList = pd.read_csv(
    FileList_Path
)

Sample_Name = str(
    FileList.iloc[0]["FileName"]
)

Sample_Path = (
    Videos_Dir
    /
    f"{Sample_Name}.avi"
)

Video_Tensor, Frame_Indices = (
    Load_And_Preprocess_Video(
        Video_Path=Sample_Path,
        Number_Of_Frames=16,
        Image_Size=224
    )
)

print(
    "Shape:",
    Video_Tensor.shape
)

print(
    "Dtype:",
    Video_Tensor.dtype
)

print(
    "Finite:",
    torch.isfinite(
        Video_Tensor
    ).all().item()
)

print(
    "Frames:",
    Frame_Indices
)

Shape: torch.Size([3, 16, 224, 224])
Dtype: torch.float32
Finite: True
Frames: [  0  11  23  34  46  57  69  80  92 103 115 126 138 149 161 173]


In [3]:
from transformers import AutoModel

MODEL_ID = "galilai-group/LeVJEPA-VideoMix-Large"

Device = torch.device("cuda:0")

Model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_safetensors=True
)

Model = Model.to(Device)

Model.eval()

print("Model loaded successfully.")
print("Device:", next(Model.parameters()).device)

Total_Parameters = sum(
    Parameter.numel()
    for Parameter in Model.parameters()
)

print(
    "Parameters:",
    f"{Total_Parameters:,}"
)

config.json:   0%|          | 0.00/576 [00:00<?, ?B/s]

configuration_levjepa.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/galilai-group/LeVJEPA-VideoMix-Large:
- configuration_levjepa.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_levjepa.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/galilai-group/LeVJEPA-VideoMix-Large:
- modeling_levjepa.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

Model loaded successfully.
Device: cuda:0
Parameters: 303,099,904


In [4]:
Input_Video = (
    Video_Tensor
    .unsqueeze(0)
    .to(Device)
)

print(
    "Input shape:",
    Input_Video.shape
)

with torch.inference_mode():

    Model_Output = Model(
        pixel_values=Input_Video
    )

print(
    "Output type:",
    type(Model_Output)
)

print(
    "Output keys:",
    Model_Output.keys()
)

print(
    "Last hidden state:",
    Model_Output.last_hidden_state.shape
)

print(
    "Pooler output:",
    Model_Output.pooler_output.shape
)

print(
    "Finite:",
    torch.isfinite(
        Model_Output.pooler_output
    ).all().item()
)

Input shape: torch.Size([1, 3, 16, 224, 224])


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


Output type: <class 'transformers.modeling_outputs.BaseModelOutput'>
Output keys: odict_keys(['last_hidden_state', 'pooler_output'])
Last hidden state: torch.Size([1, 3137, 1024])
Pooler output: torch.Size([1, 1024])
Finite: True


In [5]:
from torch.utils.data import Dataset, DataLoader

FileList = pd.read_csv(
    FileList_Path
)

Split_Values = (
    FileList["Split"]
    .astype(str)
    .str.upper()
)

Train_Data = (
    FileList[
        Split_Values == "TRAIN"
    ]
    .reset_index(drop=True)
)

Val_Data = (
    FileList[
        Split_Values == "VAL"
    ]
    .reset_index(drop=True)
)

Test_Data = (
    FileList[
        Split_Values == "TEST"
    ]
    .reset_index(drop=True)
)

EF_Mean = float(
    Train_Data["EF"].mean()
)

EF_Std = float(
    Train_Data["EF"].std()
)

print("TRAIN:", len(Train_Data))
print("VAL:", len(Val_Data))
print("TEST:", len(Test_Data))

print("EF Mean:", EF_Mean)
print("EF Std:", EF_Std)

TRAIN: 7465
VAL: 1288
TEST: 1277
EF Mean: 55.77764325112096
EF Std: 12.407189604685396


In [6]:
class EchoNetDataset(Dataset):

    def __init__(
        self,
        DataFrame,
        Videos_Dir,
        EF_Mean,
        EF_Std,
        Number_Of_Frames=16,
        Image_Size=224
    ):

        self.DataFrame = (
            DataFrame
            .reset_index(drop=True)
        )

        self.Videos_Dir = Path(
            Videos_Dir
        )

        self.EF_Mean = float(
            EF_Mean
        )

        self.EF_Std = float(
            EF_Std
        )

        self.Number_Of_Frames = (
            Number_Of_Frames
        )

        self.Image_Size = (
            Image_Size
        )


    def __len__(
        self
    ):

        return len(
            self.DataFrame
        )


    def __getitem__(
        self,
        Index
    ):

        Row = self.DataFrame.iloc[
            Index
        ]

        Video_Name = str(
            Row["FileName"]
        )

        EF = float(
            Row["EF"]
        )

        EF_Normalized = (
            EF - self.EF_Mean
        ) / self.EF_Std

        Video_Path = (
            self.Videos_Dir
            /
            f"{Video_Name}.avi"
        )

        Video_Tensor, Frame_Indices = (
            Load_And_Preprocess_Video(
                Video_Path=Video_Path,
                Number_Of_Frames=
                    self.Number_Of_Frames,
                Image_Size=
                    self.Image_Size
            )
        )

        return {
            "video":
                Video_Tensor,

            "ef":
                torch.tensor(
                    EF,
                    dtype=torch.float32
                ),

            "ef_normalized":
                torch.tensor(
                    EF_Normalized,
                    dtype=torch.float32
                ),

            "video_name":
                Video_Name,

            "frame_indices":
                torch.as_tensor(
                    Frame_Indices,
                    dtype=torch.long
                )
        }

In [7]:
Train_Dataset = EchoNetDataset(
    Train_Data,
    Videos_Dir,
    EF_Mean,
    EF_Std
)

Val_Dataset = EchoNetDataset(
    Val_Data,
    Videos_Dir,
    EF_Mean,
    EF_Std
)

Test_Dataset = EchoNetDataset(
    Test_Data,
    Videos_Dir,
    EF_Mean,
    EF_Std
)

print("Train:", len(Train_Dataset))
print("Val:", len(Val_Dataset))
print("Test:", len(Test_Dataset))

Train: 7465
Val: 1288
Test: 1277


In [8]:
Sample = Train_Dataset[0]

print(
    Sample.keys()
)

print(
    "Video:",
    Sample["video"].shape
)

print(
    "EF:",
    Sample["ef"]
)

print(
    "EF normalized:",
    Sample["ef_normalized"]
)

print(
    "Video name:",
    Sample["video_name"]
)

dict_keys(['video', 'ef', 'ef_normalized', 'video_name', 'frame_indices'])
Video: torch.Size([3, 16, 224, 224])
EF: tensor(59.1020)
EF normalized: tensor(0.2679)
Video name: 0X1002E8FBACD08477


In [9]:
import gc

try:
    del Model
except:
    pass

try:
    del Model_Output
except:
    pass

try:
    del Input_Video
except:
    pass

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU memory cleared."
)

GPU memory cleared.


In [10]:
import torch.nn as nn
from transformers import AutoModel


MODEL_ID = (
    "galilai-group/"
    "LeVJEPA-VideoMix-Large"
)


class LeVJEPA_EF_Model(nn.Module):

    def __init__(
        self,
        Model_ID
    ):

        super().__init__()

        self.Backbone = (
            AutoModel.from_pretrained(
                Model_ID,
                trust_remote_code=True,
                use_safetensors=True
            )
        )

        Hidden_Size = 1024

        self.EF_Head = nn.Linear(
            Hidden_Size,
            1
        )


    def forward(
        self,
        Pixel_Values
    ):

        Backbone_Output = (
            self.Backbone(
                pixel_values=
                    Pixel_Values
            )
        )

        Global_Feature = (
            Backbone_Output
            .pooler_output
        )

        EF_Prediction = (
            self.EF_Head(
                Global_Feature
            )
            .squeeze(-1)
        )

        return EF_Prediction


Model = LeVJEPA_EF_Model(
    MODEL_ID
)

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

In [11]:
if hasattr(
    Model.Backbone,
    "gradient_checkpointing_enable"
):

    try:

        Model.Backbone.gradient_checkpointing_enable()

        print(
            "Gradient checkpointing enabled."
        )

    except Exception as Error:

        print(
            "Gradient checkpointing unavailable:"
        )

        print(
            Error
        )

else:

    print(
        "Gradient checkpointing "
        "not supported."
    )

Gradient checkpointing unavailable:
LeVJEPAModel does not support gradient checkpointing.


In [12]:
Total_Parameters = sum(
    Parameter.numel()
    for Parameter in Model.parameters()
)

Trainable_Parameters = sum(
    Parameter.numel()
    for Parameter in Model.parameters()
    if Parameter.requires_grad
)

Frozen_Parameters = (
    Total_Parameters
    -
    Trainable_Parameters
)

print("Total:", f"{Total_Parameters:,}")
print("Trainable:", f"{Trainable_Parameters:,}")
print("Frozen:", f"{Frozen_Parameters:,}")

assert Frozen_Parameters == 0

print("\nFULL FINE-TUNING ENABLED.")

Total: 303,100,929
Trainable: 303,100,929
Frozen: 0

FULL FINE-TUNING ENABLED.


In [13]:
Device = torch.device("cuda:0")

Model = Model.to(Device)

if torch.cuda.device_count() >= 2:

    Model = nn.DataParallel(
        Model,
        device_ids=[0, 1]
    )

    print("Multi-GPU enabled.")
    print("GPU 0:", torch.cuda.get_device_name(0))
    print("GPU 1:", torch.cuda.get_device_name(1))

else:

    print("Single GPU mode.")

Multi-GPU enabled.
GPU 0: Tesla T4
GPU 1: Tesla T4


In [14]:
Batch_Size = 2

Gradient_Accumulation_Steps = 4

Effective_Batch_Size = (
    Batch_Size
    *
    Gradient_Accumulation_Steps
)

Num_Workers = 2


Train_Loader = DataLoader(
    Train_Dataset,
    batch_size=Batch_Size,
    shuffle=True,
    num_workers=Num_Workers,
    pin_memory=True,
    drop_last=False
)


Val_Loader = DataLoader(
    Val_Dataset,
    batch_size=Batch_Size,
    shuffle=False,
    num_workers=Num_Workers,
    pin_memory=True,
    drop_last=False
)


Test_Loader = DataLoader(
    Test_Dataset,
    batch_size=Batch_Size,
    shuffle=False,
    num_workers=Num_Workers,
    pin_memory=True,
    drop_last=False
)


print("Train batches:", len(Train_Loader))
print("Val batches:", len(Val_Loader))
print("Test batches:", len(Test_Loader))

print(
    "Effective batch size:",
    Effective_Batch_Size
)

Train batches: 3733
Val batches: 644
Test batches: 639
Effective batch size: 8


In [15]:
import math

from transformers import (
    get_cosine_schedule_with_warmup
)


Max_Epochs = 10

Early_Stopping_Patience = 3

Min_Improvement = 0.05


Backbone_Learning_Rate = 1e-5

Head_Learning_Rate = 1e-4

Weight_Decay = 0.01

Gradient_Clip_Norm = 1.0


Loss_Function = nn.MSELoss()


Base_Model = (
    Model.module
    if isinstance(Model, nn.DataParallel)
    else Model
)


Optimizer = torch.optim.AdamW(
    [
        {
            "params":
                Base_Model.Backbone.parameters(),

            "lr":
                Backbone_Learning_Rate
        },

        {
            "params":
                Base_Model.EF_Head.parameters(),

            "lr":
                Head_Learning_Rate
        }
    ],

    weight_decay=
        Weight_Decay
)


Optimizer_Steps_Per_Epoch = math.ceil(
    len(Train_Loader)
    /
    Gradient_Accumulation_Steps
)


Total_Optimizer_Steps = (
    Optimizer_Steps_Per_Epoch
    *
    Max_Epochs
)


Warmup_Steps = max(
    1,
    int(
        Total_Optimizer_Steps
        *
        0.05
    )
)


Scheduler = (
    get_cosine_schedule_with_warmup(
        Optimizer,
        num_warmup_steps=
            Warmup_Steps,
        num_training_steps=
            Total_Optimizer_Steps
    )
)


Scaler = torch.amp.GradScaler(
    "cuda"
)


print(
    "Optimizer steps / epoch:",
    Optimizer_Steps_Per_Epoch
)

print(
    "Maximum optimizer steps:",
    Total_Optimizer_Steps
)

print(
    "Warmup steps:",
    Warmup_Steps
)

print(
    "AMP: FP16"
)

print(
    "\nFULL FINE-TUNING SETUP READY."
)

Optimizer steps / epoch: 934
Maximum optimizer steps: 9340
Warmup steps: 467
AMP: FP16

FULL FINE-TUNING SETUP READY.
